# PySpark — AQE & Broadcast Joins for Skew, Quick Notes

<img src='https://miro.medium.com/v2/resize:fit:1100/format:webp/1*uPysuQxxiEP84Dy0kdHoLg.png' height=400/>

## Baseline problem

Without AQE or broadcast, Spark defaults to **sort-merge join**. A skewed join key means one oversized partition after the shuffle, one straggler task, long tail on the job.

## AQE (Adaptive Query Execution)

Spark 3.0+. Uses **runtime statistics** (bytes read, partition sizes, distinct key counts) — gathered *after* a shuffle stage finishes — to adjust the plan for what follows.

<img src='https://www.databricks.com/wp-content/uploads/2020/05/blog-adaptive-query-execution-2.png' height=400/>

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")   # both required for skew handling specifically
```


**Three separate optimizations, not one:**

1. **Coalesce shuffle partitions.** E.g. 200 default partitions but only 15 distinct keys → 185 would sit empty → AQE coalesces down to ~15. Fewer partitions = fewer tasks = less overhead.

<img src="https://www.databricks.com/wp-content/uploads/2020/05/blog-adaptive-query-execution-3.png" height=300/>

2. **Convert sort-merge join → broadcast join at runtime**, if post-shuffle stats reveal one side is actually small enough — even if you never explicitly asked for a broadcast.
3. **Split skewed partitions.** An oversized post-shuffle partition gets broken into smaller pieces so no single task carries a disproportionate share.

**Effect on skew — real, but partial.** Task duration example: without AQE, 11 ms to 7 sec (huge gap = skew). With AQE, 2 to 7 sec (gap narrowed, not eliminated). Job time barely moved (10.9s → 11.8s in the source example) — **AQE reduces skew, it doesn't guarantee it's solved.** Manual intervention (salting, broadcast, repartitioning) is still sometimes needed.

## Broadcast join

<img src='https://miro.medium.com/v2/resize:fit:1100/format:webp/0*hVV5E7QsM3EPSIBq.png' height=300/>

**Why sort-merge join gets skewed in the first place:** both tables are shuffled — redistributed by `hash(join_key) % shuffle_partitions`. Partitioning *by the join key* is exactly what concentrates a dominant key's rows onto one partition.

<img src="https://www.databricks.com/wp-content/uploads/2020/05/blog-adaptive-query-execution-4.png" height=300/>

**Why broadcast join avoids this — the actual mechanism:** the small (dim) table is sent in full to every executor. The large (fact) table is **never reshuffled by the join key at all** — it keeps whatever partitioning it already had (e.g. from the read, or a plain `repartition`). Each executor just probes its existing local fact partition against its local copy of the broadcast table.

So broadcast join isn't skew-resistant because "all keys are available everywhere" — it's skew-resistant because **the large table is never redistributed by the join key**, so there's no mechanism left that could concentrate one key's rows onto a single partition.

```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MB, e.g.
```

## Head-to-head (same skewed dataset)

| Approach | Time |
|---|---|
| Sort-merge join, no AQE | baseline (slow, visible straggler task) |
| AQE on | 10.9s — task gap narrowed, job time barely improved |
| Broadcast join | 3.2s — roughly ⅓ of AQE's time |

**Takeaway:** when broadcast is applicable (one side genuinely small), it beats AQE's skew handling outright — AQE still does a full shuffle-then-split, broadcast skips the shuffle entirely. AQE is the safety net for cases where broadcast isn't an option (both sides large).


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark.conf.set('spark.sql.adaptive.enabled', False)

## Read Data

In [ ]:
transaction_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/pyspark_optimization/data/data_skew/transactions.parquet/"
customer_path =  "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/pyspark_optimization/data/data_skew/customers.parquet/"

df_transactions = spark.read.format('parquet').load(transaction_path)
df_customers = spark.read.format('parquet').load(customer_path)

In [ ]:
print(df_transactions.rdd.getNumPartitions())
df_transactions.printSchema()
display(df_transactions.limit(5))

In [ ]:
print(df_customers.rdd.getNumPartitions())
df_customers.printSchema()
display(df_customers.limit(5))

## Join Skews

- First check the joins using sort-merge joins not broadcast

In [ ]:
(
    df_transactions
    .groupBy('cust_id')
    .count()
    .show(5, False)
)

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1) ## Disable Broadcast Joins

In [ ]:
df_sales = (
    df_transactions
    .join(
        df_customers,
        "cust_id",
        "inner"
    )
)

df_sales.count()

This took around 27 seconds

## Join using `AQE`

Adative Query Execution

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', True)
spark.conf.set('spark.sql.adaptive.skewedJoin.enabled', True)

In [ ]:
df_sales = (
    df_transactions
    .join(
        df_customers,
        "cust_id",
        "inner"
    )
)

df_sales.count()

This took around 20 seconds, there is improvement in the query execution

## Broadcast Joins

In [ ]:
# 10MB = 10485760 Bytes
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

In [ ]:
df_txn_details = (
    df_transactions.join(
        F.broadcast(df_customers),
        on="cust_id",
        how="inner"
    )
)

df_txn_details.count()